# Ngày 26 · Tích chập và CNN

- Tính kích thước đầu ra của tích chập theo kernel, stride, padding
- Cài conv2d và max pooling bằng vòng lặp rồi so với PyTorch
- Dựng và huấn luyện một CNN nhỏ trên ảnh chữ số

**Đọc:**
- Goodfellow, Bengio & Courville (2016), Deep Learning, Chương 9 Convolutional Networks (§9.1–9.3)
- ISLR (bản 2), §10.3 Convolutional Neural Networks
- LeCun, Bottou, Bengio & Haffner (1998), Gradient-Based Learning Applied to Document Recognition, Proceedings of the IEEE 86(11), 2278–2324
- Dumoulin & Visin (2016), A Guide to Convolution Arithmetic for Deep Learning, arXiv:1603.07285

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
try:
    import torch
    from torch import nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError('Kernel này chưa có PyTorch. Cài bằng Python của Jupyter: python -m pip install -r requirements-py30.txt') from error
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
# load_digits (UCI Optical Recognition of Handwritten Digits): 1797 ảnh 8×8, chia 80/20 phân tầng, seed 0.
_Xd, _yd = load_digits(return_X_y=True)
_itr, _iva = train_test_split(np.arange(len(_yd)), test_size=0.2, random_state=0, stratify=_yd)
X_tr = torch.tensor(_Xd[_itr] / 16, dtype=torch.float32); y_tr = torch.tensor(_yd[_itr])
X_va = torch.tensor(_Xd[_iva] / 16, dtype=torch.float32); y_va = torch.tensor(_yd[_iva])
def _loader(X, y, batch_size, seed):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                                       generator=torch.Generator().manual_seed(seed))
def _train_epoch(model, loader, loss_fn, optimizer):
    model.train(); total, n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); optimizer.step()
        total += loss.item() * len(yb); n += len(yb)
    return total / n
def _accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return float((model(X).argmax(1) == y).float().mean())


## conv2d, max pooling và kích thước đầu ra

Gợi ý: Đệm bằng F.pad(x, (p, p, p, p)). Với mỗi vị trí đầu ra (i, j): lấy vùng x[:, :, i·s : i·s + k, j·s : j·s + k], nhân với trọng số rồi cộng theo (C_in, k, k). Có thể dùng torch.einsum("nchw,ochw->no", vùng, w).

In [ ]:
def conv_output_size(n, k, stride=1, padding=0):
    raise NotImplementedError

def conv2d_naive(x, w, b, stride=1, padding=0):
    # x: (N, C_in, H, W); w: (C_out, C_in, k, k); b: (C_out,). Dùng vòng lặp trên vị trí đầu ra.
    raise NotImplementedError

def max_pool2d_naive(x, k):
    # Cửa sổ k×k, bước k, không đệm.
    raise NotImplementedError


In [ ]:
torch.manual_seed(26)
_x26, _w26, _b26 = torch.randn(2, 3, 9, 7), torch.randn(4, 3, 3, 3), torch.randn(4)
_report('d26-conv',
    _check('Công thức kích thước đầu ra', lambda: (conv_output_size(8, 3, 1, 1) == 8 and conv_output_size(28, 5, 1, 0) == 24 and conv_output_size(9, 3, 2, 1) == 5, '⌊(n + 2p − k)/s⌋ + 1.')),
    _check('conv2d khớp F.conv2d (stride 1, padding 0)', lambda: (torch.allclose(conv2d_naive(_x26, _w26, _b26), F.conv2d(_x26, _w26, _b26), atol=1e-5), 'Ảnh 9×7, 3 kênh vào, 4 kênh ra.')),
    _check('conv2d khớp F.conv2d (stride 2, padding 1)', lambda: (torch.allclose(conv2d_naive(_x26, _w26, _b26, 2, 1), F.conv2d(_x26, _w26, _b26, stride=2, padding=1), atol=1e-5), 'Ảnh không vuông.')),
    _check('max pooling khớp F.max_pool2d', lambda: (torch.allclose(max_pool2d_naive(_x26[:, :, :8, :6], 2), F.max_pool2d(_x26[:, :, :8, :6], 2)), 'Cửa sổ 2×2.')),
)


## CNN nhỏ cho ảnh chữ số 8×8

Gợi ý: Conv2d(1, 8, 3, padding=1) → ReLU → MaxPool2d(2) → Conv2d(8, 16, 3, padding=1) → ReLU → MaxPool2d(2) → Flatten → Linear(16·2·2, 10). Ảnh đầu vào: X.view(-1, 1, 8, 8). Dùng _train_epoch và _loader có sẵn.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        # x: (N, 1, 8, 8) → logits (N, 10)
        raise NotImplementedError

def fit_cnn(seed, epochs=15):
    # torch.manual_seed(seed), Adam lr = 0.01, batch 64 với _loader(..., seed). Trả (model, accuracy validation).
    raise NotImplementedError


In [ ]:
_c26 = {}
def _fit26():
    if 'r' not in _c26:
        _c26['r'] = fit_cnn(0)
    return _c26['r']
_report('d26-cnn',
    _check('Đầu ra (N, 10) và đúng 1898 tham số', lambda: ((lambda m: tuple(m(torch.zeros(3, 1, 8, 8)).shape) == (3, 10) and sum(p.numel() for p in m.parameters()) == 1898)(SmallCNN()), '80 + 1168 + 650.')),
    _check('Có tích chập và pooling', lambda: ((lambda m: sum(isinstance(x, nn.Conv2d) for x in m.modules()) == 2 and sum(isinstance(x, nn.MaxPool2d) for x in m.modules()) == 2)(SmallCNN()), 'Hai khối Conv → ReLU → MaxPool.')),
    _check('Accuracy validation ≥ 0,95 sau 15 epoch', lambda: ((lambda r: (r[1] >= .95, f'Đạt {r[1]:.3f}.'))(_fit26())[0], 'Seed 0.')),
)


## Dự án

Chạy notebook cloud Ngay_25_26_28_CNN trên GPU: huấn luyện CNN trên Fashion-MNIST, so accuracy với MLP ngày 24 và logistic ngày 22. In số tham số của hai mô hình. Sau đó chạy phần Digit Recognizer để xuất submission và so với kết quả PCA + KNN của ngày 21.